Projeto Final - Mídulo 1 - SCTEC

#Fase 4: Divisão e Balanceamento dos Dados

Separe as variáveis preditoras (X) da variável alvo (y).
Divida os dados em treino (80%) e teste (20%) utilizando o parâmetro stratify=y.
Aplique uma técnica de reamostragem (SMOTE ou Random Under Sampling) exclusivamente nos dados de treino para evitar o vazamento de dados (Data Leakage).


In [1]:
# Configuração inicial para todos os notebooks

# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuração do visual global
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("n\=== CONFIGURAÇÕES E BIBLIOTECAS IMPORTADAS COM SUCESSO ===")

<>:16: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
<>:16: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_1816\217933160.py:16: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
  print("n\=== CONFIGURAÇÕES E BIBLIOTECAS IMPORTADAS COM SUCESSO ===")


n\=== CONFIGURAÇÕES E BIBLIOTECAS IMPORTADAS COM SUCESSO ===


In [2]:
# Carregando os dados do dataset de engenharia

df_fe = pd.read_csv("../data/processed/manutencao_preditiva_fe.csv", sep=",")
df_fe.head()

,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,potencia_calc
0,M14860,M,298.1,308.6,1551.0,42.8,0,0,66382.8
1,L47181,L,298.2,308.7,1408.0,46.3,3,0,65190.4
2,L47182,L,298.1,308.5,1498.0,49.4,5,0,74001.2
3,L47183,L,300.1,310.1,1504.0,40.1,7,0,60310.4
4,L47184,L,298.2,308.7,1408.0,40.0,9,0,56320.0


In [3]:
# Separando as variáveis preditoras (x) eliminando a coluna da variável alvo (falha_maquina)

X = df_fe.drop(columns=["falha_maquina"]) # X é maiúsculo
X.head()

,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,potencia_calc
0,M14860,M,298.1,308.6,1551.0,42.8,0,66382.8
1,L47181,L,298.2,308.7,1408.0,46.3,3,65190.4
2,L47182,L,298.1,308.5,1498.0,49.4,5,74001.2
3,L47183,L,300.1,310.1,1504.0,40.1,7,60310.4
4,L47184,L,298.2,308.7,1408.0,40.0,9,56320.0


In [4]:
# Isolando apenas a variável alvo

y = df_fe["falha_maquina"] # y é minúsculo
y.head()


0    0
1    0
2    0
3    0
4    0
Name: falha_maquina, dtype: int64

In [5]:
# Dividindo a base de dados em treino (80%) e teste (20%)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Dimensões de treino (X_train): {X_train.shape}") # (quantidade de linhas, colunas)
print(f"Dimensões de teste (X_test): {X_test.shape}")


Dimensões de treino (X_train): (8000, 8)
Dimensões de teste (X_test): (2000, 8)


from sklearn.utils import resample

# 4. Balanceamento de classes via técnica de reamostragem (Random Under Sampling)
treino_completo = pd.concat([X_train, y_train], axis=1)
classe_normal = treino_completo[treino_completo['falha_maquina'] == 0]
classe_falha = treino_completo[treino_completo['falha_maquina'] == 1]

# Reduzindo a classe majoritária para o tamanho da minoritária
classe_normal_subamostrada = resample(
    classe_normal, replace=False, n_samples=len(classe_falha), random_state=42
)

treino_balancedo = pd.concat([classe_normal_subamostrada, classe_falha])
X_train_bal = treino_balancedo.drop(columns=['falha_maquina'])
y_train_bal = treino_balancedo['falha_maquina']


In [6]:
# Balanceamento de classes por reamostragem (Random Under Sampling)
# serve para evitar que o modelo "chute" a resposta mais frequente

# importando a ferramente resample
from sklearn.utils import resample

treino_completo = pd.concat([X_train, y_train], axis=1)
classe_normal = treino_completo[treino_completo['falha_maquina'] == 0]
classe_falha = treino_completo[treino_completo['falha_maquina'] == 1]

# Reduzindo a classe majoritária para o tamanho da minoritária
classe_normal_subamostrada = resample(
    classe_normal, replace=False, n_samples=len(classe_falha), random_state=42
) 

treino_balanceado = pd.concat([classe_normal_subamostrada, classe_falha])
X_train_bal = treino_balanceado.drop(columns=['falha_maquina'])
y_train_bal = treino_balanceado['falha_maquina']

y_train_bal.value_counts() # mostra a quantidade de linhas de cada uma (0 e 1). Devem ser iguais.

falha_maquina
0    271
1    271
Name: count, dtype: int64

# Fase 5: Escalonamento de Variáveis (StandardScaler)

Aplique o StandardScaler apenas nas variáveis contínuas destinadas ao modelo KNN (utilizando fit_transform no treino e transform no teste).
Mantenha os dados da Árvore de Decisão sem escalonamento, justificando no código o motivo de o algoritmo ser imune à escala dos atributos.


In [7]:
# Listando (isolando) as variáveis contínuas que possuem escalas muito distantes
colunas_continuas=["velocidade_rotacao_rpm", "torque_nm", "potencia_calc", "temperatura_ar_k", "temperatura_processo_k"]


# Importando a ferramenta StandardScaler
from sklearn.preprocessing import StandardScaler

# Aplicando o StandardScaler de forma segura para o KNN
# Serve para colocar todas as colunas numéricas na mesma escala, com média 0 e desvio padrão 1, para que a comparação seja justa
scaler = StandardScaler()
X_train_knn = X_train_bal.copy()
X_test_knn = X_test.copy()

X_train_knn[colunas_continuas] = scaler.fit_transform(X_train_knn[colunas_continuas])
X_test_knn[colunas_continuas] = scaler.transform(X_test_knn[colunas_continuas])

X_train_knn.head()

,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,potencia_calc
4497,L51677,L,1.008574,0.118924,0.048896,-0.664263,72,-0.694809
949,L48129,L,-2.391033,-2.717462,0.446172,-0.903587,56,-0.786590
6377,L53557,L,-0.173898,-0.377443,0.112179,-0.591740,96,-0.543839
649,M15509,M,-1.356370,-0.661082,0.783681,-1.186424,152,-1.035948
5179,M20039,M,1.747618,2.104394,-0.207752,-0.171110,101,-0.196038


### Justificativa: Por que a Árvore de Decisão não precisa de escala?

As variáveis para a Árvore de decisão são as geradas na etapa anterior: X_train_bal e X_test.
Elas não precisam de tratamento  StandardScaler, pois a Árvore de decisão é um algoritmo que toma decisões criando um fluxograma de perguntas de maior ou menor (regras de corte), analisando coluna a coluna, de forma isolada e, portanto, permanecendo imune a escala, magnitude ou tamanho dos números das outras colunas.

Já o KNN usa geometria e requer escalonamento dos dados para que os números grandes não esmaguem os números pequenos. 


# Fase 6: Ajuste de Parâmetros e Combate ao Overfitting 

No KNN: 
Treine o modelo variando o parâmetro n_neighbors (K) por no mínimo 3 valores ímpares (ex: K = 3, 5, 7) e registre a acurácia no treino e no teste. 
Na Árvore: Treine o modelo variando o parâmetro max_depth por no mínimo 3 limites (ex: 3, 5 e None) e registre a acurácia no treino e no teste. 
Insira um texto identificando em quais pontos ocorreu o overfitting e qual configuração garantiu a estabilidade no teste. 


In [ ]:
# Importação as ferramentas

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [12]:
# Ajuste do KNN

valores_k = [3, 5, 7]
knn_resultados = {}

print("--- Resultados do KNN ---")
for k in valores_k:
    knn_ajustado = KNeighborsClassifier(n_neighbors=k) # para instanciar o modelo com o k da vez
    knn_ajustado.fit(X_train_knn[colunas_continuas], y_train_bal) # para treinar o modelo
    # Previsões para a base teste e treino(para ver o Overfitting)
    knn_preds_teste = knn_ajustado.predict(X_test_knn[colunas_continuas])
    knn_preds_treino = knn_ajustado.predict(X_train_knn[colunas_continuas])

    # Avaliar a precisão dos dados de teste
    acc_teste = accuracy_score(y_test, knn_preds_teste)
    acc_treino = accuracy_score(y_train_bal, knn_preds_treino)
    knn_resultados[k] = {'treino': acc_treino, 'teste': acc_teste}
    print(f"K = {k} | Treino: {acc_treino:.2f} | Teste: {acc_teste:.2f}")


--- Resultados do KNN ---
K = 3 | Treino: 0.87 | Teste: 0.84
K = 5 | Treino: 0.85 | Teste: 0.87
K = 7 | Treino: 0.84 | Teste: 0.88


In [14]:
# Ajuste da Árvore de decisão

valores_depth = [3, 5, None]
tree_resultados = {}

print("--- Resultados da Árvore de Decisão ---")
for depth in valores_depth:
    tree_ajustada = DecisionTreeClassifier(max_depth=depth, random_state=42) # para instanciar o modelo
    tree_ajustada.fit(X_train_bal[colunas_continuas], y_train_bal) # para treinar o modelo
    # Previsões para a base teste e treino(para ver o Overfitting)
    tree_preds_teste = tree_ajustada.predict(X_test[colunas_continuas])
    tree_preds_treino = tree_ajustada.predict(X_train_bal[colunas_continuas])

    # Avaliar a precisão dos dados de teste
    acc_teste = accuracy_score(y_test, tree_preds_teste)
    acc_treino = accuracy_score(y_train_bal, tree_preds_treino)
    chave = 'None' if depth is None else depth
    tree_resultados[chave] = {'treino': acc_treino, 'teste': acc_teste}
    print(f"Max Depht = {chave} | Treino: {acc_treino:.2f} | Teste: {acc_teste:.2f}")


--- Resultados da Árvore de Decisão ---
Max Depht = 3 | Treino: 0.84 | Teste: 0.89
Max Depht = 5 | Treino: 0.87 | Teste: 0.89
Max Depht = None | Treino: 0.98 | Teste: 0.74


Análise:

Observando os resultados da acuracia de treino e teste, foi observado overfitting no modelo de Árvore de Decisão para o Max Depth = None, uma vez que em treino, a acuracia foi de 0.98, muito próxima a 1.00, mas, no teste, houve uma queda para 0.74, indicando perda de precisão com novos dados. As demais configurações Max Depth 3 e 5, evitaram o overfitting e garantiram a estabilidade, mantendo a acurácia de teste em 0.89.

No modelo KNN não foi observado overfitting crítico. As acurácias de treino e teste se mantiveram próximas em todas as configurações. Com K = 7 obteve-se a maior estabilidade e melhor resultado, com 
acurácia de treino 0.84 e de teste 0.88.



# Fase 7: Avaliação da Acurácia e Veredito Final 
Calcule e exiba a acurácia final do melhor KNN e da melhor árvore de decisão utilizando os dados de teste. 
Compare as taxas de acerto e escreva uma conclusão justificando qual modelo apresentou o desempenho superior no teste e deve ser adotado pela empresa.

In [16]:
# Melhor KNN (k=7)
melhor_knn = KNeighborsClassifier(n_neighbors=7)
melhor_knn.fit(X_train_knn[colunas_continuas], y_train_bal)

# Melhor Árvore (Max Depth=3)
melhor_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
melhor_tree.fit(X_train_bal[colunas_continuas], y_train_bal)

# Acurácias finais usando apenas os dados dos testes
pred_knn_final = melhor_knn.predict(X_test_knn[colunas_continuas])
pred_tree_final = melhor_tree.predict(X_test[colunas_continuas])

acc_knn_final = accuracy_score(y_test, pred_knn_final)
acc_tree_final = accuracy_score(y_test, pred_tree_final)

print(f"Acurácia final do melhor KNN (K=7): {acc_knn_final:.2f}")
print(f"Acurária final da melhor Árvore de Decisão (Max Depth=3): {acc_tree_final:.2f}")


Acurácia final do melhor KNN (K=7): 0.88
Acurária final da melhor Árvore de Decisão (Max Depth=3): 0.89


Análise:

Analisando a acurácia com os dados de teste, o modelo de Árvore de Decisão com Max Depth = 3 apresentou desempenho superior ao modelo KNN, com taxa de acerto de 0.89 contra 0.88, respectivamente.

Conclusão: 
O modelo recomendado para a empresa predizer falhas em seu equipamento de produção é o de Árvore de Decisão, justificado pela maior precisão, garantindo previsões mais confiáveis, maior simplicidade, já que não requer padronização de escalas das variáveis e utiliza lógica de fácil visualização para a tomada de decisão da equipe. 

